# fhkit.at: privacy, legal notice, sign-in and footer

Checks on the built `public/` for the fixes of 2026-10-10: PhotonQ's Google
Analytics property is gone, the legal notice is published in every locale,
sign-in uses netsnek's CMS client instead of placeholders, the contact form
posts to fhkit's own emailwerk template instead of netsnek's, and the footer
credits Netsnek e.U. with the current year and links the legal notice that
exists.

Run after `gatsby build`. Nothing here talks to the network.

In [ ]:
import datetime, html, json, os, re
from pathlib import Path

SITE = Path(os.environ.get('FHKIT_SITE_DIR', '..')).resolve()
PUBLIC = SITE / 'public'
YEAR = os.environ.get('FHKIT_EXPECT_YEAR', str(datetime.date.today().year))
LOCALES = {'de': '', 'en': 'en/', 'sl': 'sl/', 'it': 'it/', 'ja': 'ja/'}
HEADINGS = {'de': 'Impressum', 'en': 'Legal notice', 'sl': 'Impresum',
            'it': 'Note legali', 'ja': '法的表示'}

RESULTS = []

def check(name, ok, detail=''):
    RESULTS.append({'name': name, 'pass': bool(ok), 'detail': str(detail)})
    print(('PASS  ' if ok else 'FAIL  ') + name + (f'  ({detail})' if detail else ''))
    return bool(ok)

def text_of(path):
    s = path.read_text(encoding='utf8')
    s = re.sub(r'<(script|style)[^>]*>.*?</\1>', ' ', s, flags=re.S)
    s = s.replace('<!-- -->', '')
    s = re.sub(r'<[^>]+>', ' ', s)
    return re.sub(r'\s+', ' ', html.unescape(s))

FILES = [p for p in PUBLIC.rglob('*') if p.is_file()
         and p.suffix in ('.html', '.js', '.json', '.txt', '.xml')]
CONTENT = {p: p.read_text(encoding='utf8', errors='ignore') for p in FILES}

def files_with(needle):
    return sorted(str(p.relative_to(PUBLIC)) for p, s in CONTENT.items() if needle in s)

check('public/ exists and is not empty', PUBLIC.is_dir() and len(FILES) > 0, f'{len(FILES)} files')

## No Google Analytics

`G-M58K75M9PG` is photonq.org's property. It came along with the template
fork and must not appear anywhere in the build. No page may load gtag.js, and
the jaen options embedded in the bundle must carry no `googleAnalytics`
entry. jaen's own consent code still names googletagmanager.com as its
default origin, but it only runs for a configured measurement id.

In [ ]:
hits = files_with('M58K75M9PG')
check('no M58K75M9PG in public/', not hits, hits[:5])

pages = [p for p in CONTENT if p.suffix == '.html']
for needle in ['googletagmanager.com', 'gtag/js']:
    hits = sorted(str(p.relative_to(PUBLIC)) for p in pages if needle in CONTENT[p])
    check(f'no page loads {needle}', not hits, hits[:5])

hits = [str(p.relative_to(PUBLIC)) for p, s in CONTENT.items()
        if p.suffix == '.js' and 'googleAnalytics:{' in s]
check('no googleAnalytics entry in the embedded plugin options', not hits, hits[:5])

## Legal notice

Every locale serves the notice: Netsnek e.U., formerly Florian Herbert Kleber
IT, with owner, address, phone, mailbox and register court, the disclaimer and
the privacy paragraph. The heading follows the locale, the notice itself stays
German.

In [ ]:
FACTS = ['Netsnek e.U.', 'vormals Florian Herbert Kleber IT',
         'Inhaber Florian Herbert Kleber', 'Löwengasse 14 / Lokal 2', '1030 Wien',
         'Österreich', '+43 650 824 88 11', 'office@fhkit.at', 'Handelsgericht Wien',
         'Sämtliche Texte auf der Website', 'Datenschutz']

for loc, prefix in LOCALES.items():
    page = PUBLIC / prefix / 'imprint' / 'index.html'
    if not check(f'{loc}: /{prefix}imprint/ exists', page.exists()):
        continue
    text = text_of(page)
    missing = [f for f in FACTS if f not in text]
    check(f'{loc}: notice carries every fact', not missing, missing)
    check(f'{loc}: heading is {HEADINGS[loc]!r}',
          re.search(r'<h1[^>]*>' + re.escape(HEADINGS[loc]) + '</h1>',
                    page.read_text(encoding='utf8')) is not None)
    check(f'{loc}: no Google Analytics paragraph', 'Google Analytics' not in text)

## Footer

The copyright line names the current year and Netsnek e.U., the "Impressum"
link goes to the localized `/imprint/`, the address is the one the notice
gives, and the GitHub links reach a real profile.

In [ ]:
site_pages = [p for p in PUBLIC.rglob('index.html')
              if not any(s in str(p) for s in ['/cms', '/login', '/logout', '/settings',
                                                '/signup', '/emailwerk', '/admin'])]
# The credit is in the language of the page, the rights phrase the same one
# the confirmation mail uses (FooterCopyright in src/locales/messages.ts).
RIGHTS = {'de': 'Alle Rechte vorbehalten.', 'en': 'All rights reserved.',
          'it': 'Tutti i diritti riservati.', 'ja': '無断転載を禁じます。',
          'sl': 'Vse pravice pridržane.'}
def page_locale(p):
    first = p.relative_to(PUBLIC).parts[0]
    return first if first in RIGHTS and first != 'de' else 'de'
def credit(loc):
    return f'Copyright © {YEAR} Netsnek e.U. {RIGHTS[loc]}'
with_line = [p for p in site_pages if credit(page_locale(p)) in text_of(p)]
check('every site page shows the Netsnek e.U. credit in its language',
      len(with_line) == len(site_pages), f'{len(with_line)}/{len(site_pages)}')
check('the German home page reads ' + repr(credit('de')),
      credit('de') in text_of(PUBLIC / 'index.html'))

# Every built file, the bundles included: the fallback footer that carried
# these strings without ever rendering is gone from the source.
for needle in ['Werbeagentur Nico Schett', 'Copyright © 2024', 'github.com/in/kleberbaum', 'Dreilach']:
    hits = files_with(needle)
    check(f'no {needle!r} anywhere in public/', not hits, hits[:5])

for loc, prefix in LOCALES.items():
    home = (PUBLIC / prefix / 'index.html').read_text(encoding='utf8')
    check(f'{loc}: footer links /{prefix}imprint/', f'href="/{prefix}imprint/"' in home)
    check(f'{loc}: no link to the missing /impressum', 'href="/impressum"' not in home)

## Sign-in and contact form

Sign-in uses netsnek's organization, CMS client and project, not the
placeholders that ended in `Errors.App.NotFound`. The contact form posts to
fhkit's own emailwerk template, not netsnek's, and its confirmation gives the
documented phone number.

In [ ]:
js = {p: s for p, s in CONTENT.items() if p.suffix == '.js'}
def in_js(needle):
    return any(needle in s for s in js.values())

check('no placeholder Zitadel ids', not files_with('TODO_FHKIT'))
for value in ['268283382465631862@cms', '268210807970535009', '268283277977065078']:
    check(f'bundle carries {value}', in_js(value))
check("no netsnek emailwerk template id", not files_with('cmsmguuxh0038rb2pgsrpzhur'))
check('no undocumented phone number', not files_with('6642503731'))
check("bundle carries fhkit's emailwerk template id", in_js('cmv2fl2cu14ebps2p05nwpoev'))
check('a failed send offers a mail to office@fhkit.at', in_js('office@fhkit.at'))
check('confirmation carries +43 650 824 88 11', in_js('+436508248811'))

In [ ]:
passed = sum(1 for r in RESULTS if r['pass'])
total = len(RESULTS)
print(f'\n{passed}/{total} checks passed')

out = SITE / 'tests' / 'results-fhkit-privacy-legal.json'
out.write_text(json.dumps({'passed': passed, 'total': total, 'checks': RESULTS},
                          indent=1, ensure_ascii=False), encoding='utf8')
print('written to', out)

assert passed == total, f'{total - passed} check(s) failed'